In [20]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve,roc_auc_score,RocCurveDisplay, log_loss
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer,make_column_selector
import warnings
warnings.filterwarnings('ignore')
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/Diabetes")

In [21]:
diabetes = pd.read_csv("train.csv")
diabetes

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699995,699995,29,1,59,6.9,5.2,1.5,26.1,0.88,133,...,Female,Hispanic,Postgraduate,Upper-Middle,Former,Employed,0,0,0,0.0
699996,699996,46,2,72,7.7,7.7,3.8,25.5,0.85,106,...,Female,Hispanic,Graduate,Upper-Middle,Former,Employed,0,0,1,1.0
699997,699997,35,1,50,5.6,6.1,6.4,26.9,0.88,127,...,Female,White,Graduate,Middle,Never,Employed,0,0,0,1.0
699998,699998,49,2,70,5.7,6.9,4.7,25.2,0.86,116,...,Female,White,Highschool,Lower-Middle,Never,Retired,0,0,0,1.0


In [22]:
diabetes.drop(['id','ethnicity','education_level','income_level','employment_status'],axis=1,inplace=True)

In [23]:
diabetes.isna().sum()

age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
smoking_status                        0
family_history_diabetes               0
hypertension_history                  0
cardiovascular_history                0
diagnosed_diabetes                    0
dtype: int64

In [24]:
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 21 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   age                                 700000 non-null  int64  
 1   alcohol_consumption_per_week        700000 non-null  int64  
 2   physical_activity_minutes_per_week  700000 non-null  int64  
 3   diet_score                          700000 non-null  float64
 4   sleep_hours_per_day                 700000 non-null  float64
 5   screen_time_hours_per_day           700000 non-null  float64
 6   bmi                                 700000 non-null  float64
 7   waist_to_hip_ratio                  700000 non-null  float64
 8   systolic_bp                         700000 non-null  int64  
 9   diastolic_bp                        700000 non-null  int64  
 10  heart_rate                          700000 non-null  int64  
 11  cholesterol_total         

In [25]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ss = StandardScaler()

In [26]:
transformer = ColumnTransformer(transformers=[('SS',ss,make_column_selector(dtype_include=[float,int])),
                                              ('OHE',ohe,make_column_selector(dtype_include=object))
                                             ],remainder='passthrough',verbose_feature_names_out=False)

In [27]:
diabetes['diagnosed_diabetes'] = diabetes['diagnosed_diabetes']
X, y = diabetes.drop('diagnosed_diabetes', axis = 1), diabetes['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26, stratify = diabetes['diagnosed_diabetes'])

In [28]:
X_train_trf = transformer.fit_transform(X_train)
X_test_trf = transformer.transform(X_test)

In [29]:
C = np.linspace(0.001, 10, 30)
scores = []
for j in tqdm(C):
    lr = LogisticRegression(solver='lbfgs', C = j)
    lr.fit(X_train_trf, y_train)
    y_pred = lr.predict_proba(X_test_trf)
    scores.append(['Lbfgs',j, log_loss(y_test, y_pred)])

100%|███████████████████████████████████████████| 30/30 [00:39<00:00,  1.31s/it]


In [30]:
df_scores = pd.DataFrame(scores, columns = ['Solver', 'C', 'Log Loss Score'])

In [31]:
df_scores.sort_values('Log Loss Score', ascending=True)

,Solver,C,Log Loss Score
1,Lbfgs,0.345793,0.604180
2,Lbfgs,0.690586,0.604180
3,Lbfgs,1.035379,0.604180
4,Lbfgs,1.380172,0.604180
5,Lbfgs,1.724966,0.604180
6,Lbfgs,2.069759,0.604180
7,Lbfgs,2.414552,0.604180
8,Lbfgs,2.759345,0.604180
9,Lbfgs,3.104138,0.604180
10,Lbfgs,3.448931,0.604180


In [32]:
tst_diabetes = pd.read_csv("test.csv")
tst_diabetes

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,111,Female,White,Highschool,Middle,Former,Employed,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,145,Female,White,Highschool,Middle,Never,Unemployed,0,0,0
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,184,Male,White,Highschool,Low,Never,Employed,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,128,Male,White,Graduate,Middle,Former,Employed,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,133,Male,White,Graduate,Low,Current,Unemployed,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299995,999995,59,3,185,6.3,7.3,4.4,22.8,0.81,108,...,126,Male,White,Highschool,Upper-Middle,Former,Employed,1,0,0
299996,999996,50,2,25,5.8,7.8,4.5,29.6,0.93,112,...,112,Male,Asian,Postgraduate,Lower-Middle,Never,Employed,0,0,0
299997,999997,63,1,252,5.2,7.5,8.5,25.1,0.77,129,...,135,Female,White,Highschool,Middle,Never,Employed,0,0,0
299998,999998,48,3,72,4.9,6.9,1.8,27.7,0.89,121,...,138,Male,White,Highschool,Low,Current,Retired,0,1,0


In [33]:
tst_diabetes.drop(['id','ethnicity','education_level','income_level','employment_status'],axis=1,inplace=True)

In [34]:
bm = LogisticRegression(solver = "lbfgs", C = 0.345793)
X_trans = transformer.fit_transform(X)
bm.fit(X_trans, y)

LogisticRegression(C=0.345793)

In [35]:
tst_diabetes = transformer.transform(tst_diabetes)

In [36]:
y_pred = bm.predict(tst_diabetes)

In [37]:
y_pred

array([1., 1., 1., ..., 0., 1., 1.])

In [40]:
y_pred_prob = bm.predict_proba(tst_diabetes)

In [45]:
ss = pd.read_csv("sample_submission.csv")
ss["diagnosed_diabetes"] = y_pred_prob[:, 1]
ss.to_csv("sb_log.csv", index = False)